# Crystalite — Complete CFG Run (all datasets)

End-to-end, parameterized pipeline:
1. Setup (clone, `uv sync`, data, base checkpoint, phase diagram)
2. **Adapter fine-tune** a property conditioner per dataset (mp20, alex_mp20)
3. **Guidance sweep** — conditioning adherence vs guidance scale (the headline result)
4. **Quality eval** — validity / novelty / SUN / Wasserstein / thermo
5. **Aggregate** all results into tables + save to Drive

**Runtime → Change runtime type → GPU.** Set `QUICK_MODE=True` (in the config cell) for a fast
full-pipeline smoke, `False` for the real run. The driver is **idempotent** — re-running skips
datasets whose checkpoint already exists, so a disconnect doesn't cost you finished work.


## 1. Setup


In [ ]:
# Clone the branch with the CFG code (use a token URL if the repo is private).
!git clone --branch cfg-optimized https://github.com/Blizzard57/crystalite-cfg.git
%cd crystalite-cfg


In [ ]:
# Install deps via the repo's uv + lockfile (correct Python 3.12 + CUDA torch).
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ['PATH']
!uv python install 3.12
!uv sync


In [ ]:
# Sanity: GPU visible + conditioning unit tests pass.
!uv run python -c "import torch; print('cuda:', torch.cuda.is_available())"
!uv run --group dev python -m pytest tests/test_property_conditioning.py -q


In [ ]:
# (Optional) Mount Drive to persist/restore outputs across sessions.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # To resume a prior run, uncomment:
    # !unzip -o /content/drive/MyDrive/crystalite_outputs.zip -d /
except Exception as e:
    print('Drive not mounted (fine to skip):', e)


In [ ]:
# Datasets, base checkpoint, and phase diagram.
# mp20 (~135 MB) is fast; alex_mp20 (~700 MB) is large and slow to preprocess the first time.
!uv run python src/data/download_datasets.py --datasets mp20 alex_mp20 --out data

# Released DNG base checkpoint (MP20, pca16) -> ./best.pt  (adapter base for fine-tuning)
!uv run python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='joshrosie/crystalite-datasets', filename='best.pt', repo_type='dataset', local_dir='.')"

# MP convex hull for thermo/SUN (gzip; the loader streams it, no need to decompress).
import os
os.makedirs('data/mp20/hull', exist_ok=True)
!curl -L --fail --retry 5 -C - -o data/mp20/hull/2023-02-07-ppd-mp.pkl.gz https://ndownloader.figshare.com/files/48241624
!ls -la best.pt data/mp20/hull/


## 2. Configuration

`QUICK_MODE=True` runs a short fine-tune + small sample counts to validate the whole pipeline
in well under an hour. Set it `False` for the real run (much longer — use Drive backup).


In [ ]:
import os, subprocess, json
from pathlib import Path

QUICK_MODE = True   # True = fast smoke of the full pipeline; False = real run.

# Architecture flags — MUST match the released best.pt (adapter load is strict on non-conditioner params).
MODEL_FLAGS = (
    '--type_encoding subatomic_tokenizer_pca_16 --d_model 512 --n_heads 16 --n_layers 14 '
    '--use_edge_bias --edge_bias_n_freqs 12 --edge_bias_hidden_dim 256 --edge_bias_n_rbf 32 '
    '--lattice_embed_mode mlp --lattice_repr ltri --nmax 20'
)
PPD = 'data/mp20/hull/2023-02-07-ppd-mp.pkl.gz'

# Per-dataset experiment matrix. Sweep targets are the directly-verifiable properties
# (space_group, chemical_system); scalars are still trained but need a predictor to score.
DATASETS = {
    'mp20': {
        'data_root': 'data/mp20',
        'cond_properties': 'band_gap space_group energy_above_hull formation_energy_per_atom',
        'sweep_targets': ['space_group=225'],
    },
    'alex_mp20': {
        'data_root': 'data/alex_mp20',
        'cond_properties': 'space_group chemical_system dft_band_gap dft_bulk_modulus dft_mag_density energy_above_hull hhi_score ml_bulk_modulus',
        'sweep_targets': ['space_group=225', 'chemical_system=Li-O'],
    },
}

STEPS       = 6000 if QUICK_MODE else 40000
NUM_SAMPLES = 256  if QUICK_MODE else 1024
SWEEP_N     = 256
SCALES      = '0 1 2 4'

ENV = {**os.environ, 'PYTHONPATH': '.'}
def run(cmd):
    print('\n$ ' + cmd + '\n', flush=True)
    if subprocess.run(cmd, shell=True, env=ENV).returncode != 0:
        raise RuntimeError('FAILED: ' + cmd)

print('QUICK_MODE =', QUICK_MODE, '| STEPS =', STEPS, '| NUM_SAMPLES =', NUM_SAMPLES)


## 3. Phase 1 — Adapter fine-tune a conditioner per dataset

Idempotent: skips a dataset if its `final.pt` already exists. `--ckpt_every 1000` saves
`step_latest.pt` periodically so a disconnect still leaves an evaluable checkpoint.


In [ ]:
for name, cfg in DATASETS.items():
    out  = f'outputs/cond_{name}'
    ckpt = f'{out}/checkpoints/final.pt'
    if Path(ckpt).exists():
        print(f'[skip] {name}: {ckpt} already exists'); continue
    run(
        'uv run python src/train_crystalite.py '
        f'--data_root {cfg["data_root"]} --dataset_name {name} --output_dir {out} '
        f'{MODEL_FLAGS} --bf16 --batch_size 64 '
        '--loss_weights 16 150 5 --coord_loss_mode frac_mse '
        '--sigma_data_type 0.3 --sigma_data_coord 0.3 --sigma_data_lattice 0.3 '
        f'--cond_properties {cfg["cond_properties"]} --cond_p_uncond 0.1 '
        '--adapter_pretrained best.pt --ema_decay 0.0 '
        f'--ckpt_every 1000 --ckpt_latest_only --max_steps {STEPS} --sample_frequency 0 --no_wandb'
    )


## 4. Phase 2 — Guidance sweep (conditioning adherence)

For each verifiable target, generates at guidance scales `0 1 2 4` and scores `match_rate`.
`w=0` is the unconditional baseline; conditioning works if `match_rate` climbs clearly above it.


In [ ]:
for name, cfg in DATASETS.items():
    out  = f'outputs/cond_{name}'
    ckpt = f'{out}/checkpoints/final.pt'
    if not Path(ckpt).exists():
        print(f'[skip] {name}: no checkpoint'); continue
    for tgt in cfg['sweep_targets']:
        slug = tgt.replace('=', '').replace('-', '')
        run(
            f'uv run python scripts/sweep_guidance.py --checkpoint {ckpt} '
            f'--dataset_name {name} --data_root {cfg["data_root"]} '
            f'--target {tgt} --guidance_scales {SCALES} '
            f'--num_samples {SWEEP_N} --sample_num_steps 150 --sample_mode regular --bf16 '
            f'--output_dir {out}/sweep_{slug}'
        )


## 5. Phase 3 — Generation quality eval (per dataset)

Unconditional sampling from each conditioned checkpoint: validity / uniqueness / novelty /
SUN / MSUN / Wasserstein / CHGNet thermo. Writes `metrics.json` per dataset.


In [ ]:
for name, cfg in DATASETS.items():
    out  = f'outputs/cond_{name}'
    ckpt = f'{out}/checkpoints/final.pt'
    if not Path(ckpt).exists():
        print(f'[skip] {name}: no checkpoint'); continue
    run(
        f'uv run python src/eval_crystalite_ckpt.py --checkpoint {ckpt} '
        f'--num_samples {NUM_SAMPLES} --sample_mode regular --sample_num_steps 150 '
        '--compute_novelty --compute_wasserstein --compute_structure_stats '
        f'--thermo_count 256 --thermo_mlip chgnet --thermo_ppd_mp {PPD} '
        f'--report_dir {out}/eval_reports --run_name quality'
    )


## 6. Phase 4 — Aggregate all results


In [ ]:
import glob, json
import pandas as pd

# --- Generation quality ---
qcols = ['valid_rate','comp_valid_rate','struct_valid_rate','unique_rate','novel_rate',
         'un_rate','SUN','MSUN','eval/thermo_stable_rate','eval/thermo_metastable_rate',
         'eval/thermo_e_above_hull_mean','sample_dist/wdist_density_atomic','sample_dist/wdist_nary']
qrows = []
for name in DATASETS:
    p = f'outputs/cond_{name}/eval_reports/quality/metrics.json'
    if Path(p).exists():
        m = json.load(open(p))['metrics']
        qrows.append({'dataset': name, **{c: m.get(c) for c in qcols}})
quality_df = pd.DataFrame(qrows)
print('===== GENERATION QUALITY ====='); display(quality_df)

# --- Conditioning adherence (guidance sweeps) ---
srows = []
for f in sorted(glob.glob('outputs/cond_*/sweep_*/sweep_summary.json')):
    parts = f.split('/')
    dataset, sweep = parts[1].replace('cond_', ''), parts[2]
    for r in json.load(open(f)):
        srows.append({'dataset': dataset, 'sweep': sweep,
                      'guidance_scale': r.get('guidance_scale'),
                      'match_rate': r.get('space_group/match_rate'),
                      'sg_mae': r.get('space_group/mae'),
                      'chemsys_exact': r.get('chemical_system/exact_rate'),
                      'chemsys_subset': r.get('chemical_system/subset_rate'),
                      'buildable_rate': r.get('buildable_rate')})
adherence_df = pd.DataFrame(srows)
print('===== CONDITIONING ADHERENCE ====='); display(adherence_df)

# Save consolidated CSVs.
os.makedirs('outputs/_results', exist_ok=True)
quality_df.to_csv('outputs/_results/quality.csv', index=False)
adherence_df.to_csv('outputs/_results/adherence.csv', index=False)
print('\n[save] outputs/_results/quality.csv  outputs/_results/adherence.csv')


## 7. Save everything to Google Drive


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !zip -r -q /content/drive/MyDrive/crystalite_outputs.zip /content/crystalite-cfg/outputs
    print('Saved -> MyDrive/crystalite_outputs.zip')
except Exception as e:
    print('Drive save skipped:', e)


## Notes

- **Scaling up:** set `QUICK_MODE=False` (40k steps, 1024 eval samples). Real runs take hours;
  rely on Drive backup + the idempotent driver to resume after disconnects.
- **Reading the sweep:** for each `(dataset, sweep)`, compare `match_rate` at `w>0` to `w=0`
  (the unconditional baseline). A clear rise = working conditioning; a drop at high `w` = over-guidance.
- **Scalar properties** (band_gap, bulk_modulus, mag_density, …) are trained but not auto-scored
  here — adherence for those needs an external property predictor or DFT.
- **mpts_52 / LeMat-Bulk** are not in this matrix: mpts_52 is CSP/test-oriented (nmax=52) and
  LeMat-Bulk isn't wired into this DNG pipeline.
